# SEC MD&A Pipeline — Google Colab Pro

End-to-end pipeline for `mda_merged_sic_naics_24_25.parquet`:

```
Documents
   ↓
Extract Tables  →  SQLite (cik, ein, sic, naics3, year, extracted_table1, extracted_table2, ...)
   ↓
Cleaned Text (tables removed)
   ↓
Chunk → Embed → Qdrant (text only, 15% overlap, MD&A-section-aware)
```

Qdrant runs in **local-file mode** so no separate server is needed in Colab.
If you keep the `qdrant_db/` and `sec_tables.db` files on Google Drive, the index persists across sessions.

## 0. Runtime check

In [1]:
import platform, sys, os
print('Python   :', sys.version.split()[0])
print('Platform :', platform.platform())
try:
    import torch
    print('Torch    :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU      :', torch.cuda.get_device_name(0))
except ImportError:
    print('Torch    : not installed yet (will be installed below)')


Python   : 3.12.13
Platform : Linux-6.6.122+-x86_64-with-glibc2.35
Torch    : 2.12.0+cu130 | CUDA available: True
GPU      : NVIDIA L4


## 1. Install dependencies

Colab Pro comes with `pandas`, `numpy`, `torch`, and `pyarrow` already installed, so only `sentence-transformers` and `qdrant-client` typically need to be added.  The `--quiet` flag keeps output short.

In [1]:
!uv pip install --quiet --upgrade \
    'numpy==1.26.4' \
    'sentence-transformers>=2.6.0' \
    'qdrant-client>=1.9.0' \
    'pyarrow>=14.0' \
    'pandas>=2.0'

In [12]:
!pip show qdrant-client

Name: qdrant-client
Version: 1.18.0
Summary: Client library for the Qdrant vector search engine
Home-page: https://github.com/qdrant/qdrant-client
Author: Andrey Vasnetsov
Author-email: andrey@qdrant.tech
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: grpcio, httpx, numpy, portalocker, protobuf, pydantic, urllib3
Required-by: 


In [8]:
#@title Numpy version has to be 1.26.4 and not >2 --- 'Restart session' if this cell fails and run again

import numpy
if numpy.__version__ != '1.26.4':
    print(f"Current numpy version {numpy.__version__} is incorrect. Installing 1.26.4...")
    !uv pip uninstall -y numpy
    !uv pip install numpy==1.26.4
    import sys
    sys.exit()  # Force stop execution
else:
    print("Numpy version is correct (1.26.4)")


Numpy version is correct (1.26.4)


In [1]:
!uv pip install --quiet --upgrade \
    'sentence-transformers>=2.6.0' \
    'qdrant-client>=1.9.0' \
    'pyarrow>=14.0' \
    'pandas>=2.0'


In [7]:
import numpy as np
print(np.__version__)

1.26.4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os, sys, shutil, zipfile

# Where the package should end up (the working dir, so `import mda_pipeline` works)
TARGET = '/content/drive/MyDrive/mda/test_impl/mda_pipeline/'
BASEDIR = '/content/drive/MyDrive/mda/test_impl/'

assert os.path.isdir(TARGET), (
    f'Could not find mda_pipeline/.'
)

## 2. Mount Google Drive (optional)

Skip this cell if you have uploaded the parquet directly with the Colab file panel.  Otherwise, point `DATA_DIR` at wherever your parquet lives on Drive.

In [ ]:
USE_DRIVE = True   # set False if you uploaded the parquet manually

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Point at the folder containing the parquet on your Drive:
    DATA_DIR = '/content/drive/MyDrive/sec_mda'
else:
    DATA_DIR = '/content'

import os
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)


## 3. Copy/clone the `mda_pipeline` package into Colab

Easiest path: zip the `mda_pipeline/` folder on your local machine, upload via the Colab file panel, and unzip it.  The cell below auto-handles a few common layouts.

In [ ]:
import os, sys, shutil, zipfile

# Where the package should end up (the working dir, so `import mda_pipeline` works)
TARGET = '/content/drive/MyDrive/mda/test_impl/mda_pipeline/'
BASEDIR = '/content/drive/MyDrive/mda/test_impl/'
if not os.path.isdir(TARGET):
    # 1) Look for an uploaded zip
    for cand in ['/content/mda_pipeline.zip', os.path.join(DATA_DIR, 'mda_pipeline.zip')]:
        if os.path.isfile(cand):
            print('Unzipping', cand)
            with zipfile.ZipFile(cand) as z:
                z.extractall('/content/')
            break
    else:
        # 2) Look for the package already on Drive
        drive_pkg = os.path.join(DATA_DIR, 'mda_pipeline')
        if os.path.isdir(drive_pkg):
            print('Copying from', drive_pkg)
            shutil.copytree(drive_pkg, TARGET)

assert os.path.isdir(TARGET), (
    f'Could not find mda_pipeline/. Upload mda_pipeline.zip to {DATA_DIR} '
    'or to the Colab file panel and re-run this cell.'
)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('mda_pipeline ready at', TARGET)


In [4]:
import os, sys
# After installing new versions of numpy/pandas, a Runtime Restart is REQUIRED.
# Go to Runtime -> Restart runtime, then run this cell.

BASEDIR = '/content/drive/MyDrive/mda/test_impl/'
sys.path.append(BASEDIR)


## 4. Configure paths

These four paths are the only thing you usually need to change.

In [5]:
PARQUET_PATH = os.path.join(BASEDIR, 'mda_merged_sic_naics_24_25.parquet')
QDRANT_PATH  = os.path.join(BASEDIR, 'qdrant_db')
SQLITE_PATH  = os.path.join(BASEDIR, 'sec_tables.db')
COLLECTION   = 'sec_mda'

assert os.path.isfile(PARQUET_PATH), (
    f'Parquet not found at {PARQUET_PATH}. Upload it (or update DATA_DIR).'
)
print('parquet :', PARQUET_PATH)
print('qdrant  :', QDRANT_PATH)
print('sqlite  :', SQLITE_PATH)


parquet : /content/drive/MyDrive/mda/test_impl/mda_merged_sic_naics_24_25.parquet
qdrant  : /content/drive/MyDrive/mda/test_impl/qdrant_db
sqlite  : /content/drive/MyDrive/mda/test_impl/sec_tables.db


## 5. (Optional) Quick smoke test on a small sample

Run the entire pipeline on the first 20 filings to validate everything works on this runtime before processing the full parquet.

In [9]:

from mda_pipeline import RAGPipeline

smoke = RAGPipeline(
    parquet_path=PARQUET_PATH,
    collection_name='sec_mda_smoke',
    chunk_size_tokens=512,
    overlap_percent=15,
    qdrant_path='/content/drive/MyDrive/mda/test_impl/qdrant_smoke',
    sqlite_path='/content/drive/MyDrive/mda/test_impl/sec_tables_smoke.db',
    ingest_limit=20,
    recreate_collection=True,
    embedding_batch_size=64,
)

_ = smoke.build()
# The reload above ensures the query_encoder=None fix is active here
results = smoke.query('What are the principal risk factors?', top_k=3)
display(results)
smoke.close()

ImportError: cannot load module more than once per process

ImportError: numpy._core.multiarray failed to import

In [8]:
import os

file_path = os.path.join(BASEDIR, 'mda_pipeline', 'qdrant_store.py')

if os.path.exists(file_path):
    print(f"Checking file: {file_path}\n")
    with open(file_path, 'r') as f:
        content = f.readlines()

    found_search_call = False
    found_query_call = False
    for i, line in enumerate(content):
        if "self.client.search(" in line:
            print(f"Line {i+1}: {line.strip()} <- 'search' call found")
            found_search_call = True
        if "self.client.query(" in line:
            print(f"Line {i+1}: {line.strip()} <- 'query' call found")
            found_query_call = True

    if not found_search_call and found_query_call:
        print("\nSuccess: 'search' calls appear to be replaced with 'query' calls.")
    elif found_search_call:
        print("\nWarning: 'search' calls are still present. Please update the file.")
    else:
        print("\nCould not confirm the change. Please ensure the file was correctly updated.")

else:
    print(f"Error: File not found at {file_path}")

print("\nEven if the file is correct, remember to restart your Colab runtime (Runtime -> Restart runtime) after making file changes on Drive for them to take effect, and then re-run all cells.")

Checking file: /content/drive/MyDrive/mda/test_impl/mda_pipeline/qdrant_store.py

Line 157: hits = self.client.query( <- 'query' call found

Success: 'search' calls appear to be replaced with 'query' calls.

Even if the file is correct, remember to restart your Colab runtime (Runtime -> Restart runtime) after making file changes on Drive for them to take effect, and then re-run all cells.


In [6]:
import importlib
import mda_pipeline
import mda_pipeline.qdrant_store
import mda_pipeline.pipeline

# Force reload the modified modules
importlib.reload(mda_pipeline.qdrant_store)
importlib.reload(mda_pipeline.pipeline)
importlib.reload(mda_pipeline)

print("Modules reloaded. Re-initializing smoke test...")

# Re-run the smoke test initialization
from mda_pipeline import RAGPipeline

smoke = RAGPipeline(
    parquet_path=PARQUET_PATH,
    collection_name='sec_mda_smoke_v2',
    chunk_size_tokens=512,
    overlap_percent=15,
    qdrant_path='/content/drive/MyDrive/mda/test_impl/qdrant_smoke_v2',
    sqlite_path='/content/drive/MyDrive/mda/test_impl/sec_tables_smoke_v2.db',
    ingest_limit=20,
    recreate_collection=True,
    embedding_batch_size=64,
)

smoke.build()
# This call triggered the error previously
results = smoke.query('What are the principal risk factors?', top_k=3)
display(results)

Modules reloaded. Re-initializing smoke test...
[tables] SQLite ready at /content/drive/MyDrive/mda/test_impl/sec_tables_smoke_v2.db
[embed] sentence-transformers not installed — using random fallback embeddings (NOT for production)
[qdrant] Opening local DB at /content/drive/MyDrive/mda/test_impl/qdrant_smoke_v2
[qdrant] Creating collection 'sec_mda_smoke_v2' (size=384, distance=COSINE)

===================== BUILDING SEC MD&A PIPELINE =====================
parquet         : /content/drive/MyDrive/mda/test_impl/mda_merged_sic_naics_24_25.parquet
qdrant          : /content/drive/MyDrive/mda/test_impl/qdrant_smoke_v2
sqlite          : /content/drive/MyDrive/mda/test_impl/sec_tables_smoke_v2.db
embedding model : sentence-transformers/all-MiniLM-L6-v2
chunk size      : 2048 chars (512 tokens)
overlap         : 15% (307 chars)

STEP 1: DATA INGEST
[ingest] Loading parquet: /content/drive/MyDrive/mda/test_impl/mda_merged_sic_naics_24_25.parquet
[ingest] Loaded 20 rows
[ingest] Columns: ['ac

TypeError: QdrantFastembedMixin.query() missing 1 required positional argument: 'query_text'

In [11]:
import traceback
from mda_pipeline import RAGPipeline

# 1. Re-initialize the smoke test object
smoke = RAGPipeline(
    parquet_path=PARQUET_PATH,
    collection_name='sec_mda_smoke_v2',
    chunk_size_tokens=512,
    overlap_percent=15,
    qdrant_path='/content/drive/MyDrive/mda/test_impl/qdrant_smoke_v2',
    sqlite_path='/content/drive/MyDrive/mda/test_impl/sec_tables_smoke_v2.db',
    ingest_limit=20,
    recreate_collection=False, # Use existing since we already built it
    embedding_batch_size=64,
)

try:
    # 2. Attempt the query
    query_str = 'What are the principal risk factors?'
    print(f"Executing query: {query_str}")

    results = smoke.query(query_str, top_k=3)
    display(results)
except TypeError as e:
    print(f"Caught expected error: {e}")
    traceback.print_exc()
    print("\nFIX: This error confirms that the internal Qdrant client 'query' method is shadowed by FastembedMixin.")
    print("You must update your 'qdrant_store.py' file to pass 'query_vector=' as a keyword argument.")

ImportError: cannot load module more than once per process

ImportError: numpy._core.multiarray failed to import

In [7]:
import inspect

# Check if the smoke object exists and inspect its client
try:
    client = smoke.vector_store.client
    print(f"QdrantClient class: {type(client)}")

    # Check for the existence of the internal query_encoder
    # In newer qdrant-client versions, this is usually stored in _query_encoder
    has_encoder = getattr(client, '_query_encoder', 'Unknown')
    print(f"Internal query_encoder state: {has_encoder}")

    # Verify the source code of the initialization from the perspective of the runtime
    import mda_pipeline.qdrant_store
    print("\n--- Runtime Source Code Check ---")
    lines = inspect.getsourcelines(mda_pipeline.qdrant_store.QdrantVectorStore.__init__)[0]
    for line in lines:
        if "QdrantClient(" in line:
            print(f"Initialization line in memory: {line.strip()}")
except NameError:
    print("Variable 'smoke' not found. Please run the previous cell first.")
except Exception as e:
    print(f"Inspection failed: {e}")

QdrantClient class: <class 'qdrant_client.qdrant_client.QdrantClient'>
Internal query_encoder state: Unknown

--- Runtime Source Code Check ---


## 6. Build the full pipeline

On a Colab Pro T4 GPU this typically processes ~20 chunks/second through the embedder.  Total build time scales linearly with the parquet size.

In [ ]:
from mda_pipeline import RAGPipeline

pipe = RAGPipeline(
    parquet_path=PARQUET_PATH,
    collection_name=COLLECTION,
    chunk_size_tokens=512,       # ~2048 chars per chunk
    overlap_percent=15,          # ~307 chars overlap
    embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    qdrant_path=QDRANT_PATH,
    sqlite_path=SQLITE_PATH,
    recreate_collection=False,    # set True to wipe & rebuild
    embedding_batch_size=128,     # ~128 fits comfortably on T4/A100
    qdrant_batch_size=512,
)
stats = pipe.build()
stats


## 7. Run example RAG queries (Qdrant — text only)

In [ ]:
queries = [
    'What are the principal risk factors?',
    'How does the company describe its liquidity and capital resources?',
    'What is the impact of inflation on operations?',
    'Describe the critical accounting policies used by management.',
    'What forward-looking statements are made about future revenue?',
]
for q in queries:
    pipe.query(q, top_k=3)


### Filtered queries

The Qdrant payload contains every column from the parquet (year, naics3, naics17Title, cik, ein, sic, address, company name, …) so you can filter searches:

In [ ]:
pipe.query('supply chain disruptions',
          top_k=5,
          filter_dict={'year': '2024'})

pipe.query('cybersecurity threats',
          top_k=5,
          filter_dict={'naics3': '518'})  # data processing / hosting


## 8. Query extracted tables (SQLite)

Tables removed from the MD&A text live in SQLite, keyed on `cik`, `ein`, `sic`, `naics3`, `year`.

In [ ]:
# Long-form: one row per extracted table
rows = pipe.query_tables(year='2024', limit=5)
for r in rows:
    print(f"{r['table_id']} | cik={r['cik']} naics3={r['naics3']} | {r['table_title'][:60]}")
    if r['table_data']:
        print('  first row:', r['table_data'][0])


In [ ]:
# Wide-form: one row per filing with extracted_table1, extracted_table2, ...
import json
if rows:
    doc_id = rows[0]['source_document_id']
    wide   = pipe.table_store.get_filing_row(doc_id)
    print('source_document_id:', wide['source_document_id'])
    print('n_tables          :', wide['n_tables'])
    for k, v in wide.items():
        if k.startswith('extracted_table') and v:
            data = json.loads(v)
            print(f"  {k}: title={data['title'][:60]!r}, rows={len(data['data'])}")


## 9. Inspect pipeline state

In [ ]:
info = pipe.info()
for k, v in info.items():
    print(f'{k}: {v}')


## 10. Persist to Drive (optional)

If `DATA_DIR` is on Google Drive, both `qdrant_db/` and `sec_tables.db` are *already* on Drive — Colab writes through to the mount.  Otherwise you can copy them now:

In [ ]:
import shutil

# Example: copy a locally-built index back to Drive
# shutil.copytree('/content/qdrant_db', '/content/drive/MyDrive/sec_mda/qdrant_db')
# shutil.copy('/content/sec_tables.db', '/content/drive/MyDrive/sec_mda/sec_tables.db')
print('Done.')


In [ ]:
# Close handles cleanly
pipe.close()
